
<center><ins><h1>Zeta Potential</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Zeta potential (ζ-potential) is the electrical potential at the slipping plane. This plane is the interface which separates mobile fluid from fluid that remains attached to the surface.</li> 
    <li>The usual units are millivolts (mV).</li> 
    <li>Files are imported over .xlsx in individual experiments but combined samples.</li>
</ul>

<center>
<img src="../images/zeta-potential_device.png" alt="Device" width="225" height="300">
<img src="../images/zeta-potential_diagram.png" alt="Diagram" width="357" height="300">
<img src="../images/zeta-potential_example-graph.png" alt="Example Graph" width="410" height="300">
</center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Import Statements

In [1]:
import os, sys

WORKSPACE = os.path.abspath(os.path.join(os.getcwd(), ".."))
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

from scripts import data_helper, plot_helper
import pandas as pd

CONFIG = data_helper.load_config()
META_DATA = data_helper.load_meta_data(CONFIG)
plot_helper.set_config(CONFIG)

# Variables

In [2]:

# Show numeric output in decimal format e.g., 2.15
pd.options.display.float_format = '{:,.2f}'.format

# Functions

In [3]:
def custom_zepot_import(files, sample_col, value_cols):
    master_df = pd.concat([pd.read_excel(file) for file in files], ignore_index = True)
    master_df.columns = data_helper.formatting_strings(master_df.columns) # All columns uppercase

    raw_df = pd.DataFrame()
    # add data value(s) into clean data frame
    for index, name in enumerate(value_cols):
        raw_df.insert(index, name, master_df[name])

    # divide and add sample information into clean data frame
    for index, name in enumerate(META_DATA["SAMPLE_INFORMATION"].dropna()):
        new_column = master_df[sample_col].str.split("_").str[index]
        raw_df.insert(index, name, new_column)
    
    # Fill NaN values with 0 for samples with no data
    raw_df.fillna(0, inplace = True)
    
    return raw_df

# Raw Import

In [4]:
# Define data path and get all associated files
data_dir = data_helper.get_instrument_path(CONFIG, "zetasizer")
files = data_helper.create_filelist(data_dir, skip=" ", config=CONFIG)

# Import raw data frame
raw_df = custom_zepot_import(files, 
                             sample_col = "SAMPLE_NAME",
                             value_cols = ["ZETA_POTENTIAL_MV"])
# Convert data types
raw_df = data_helper.convert_types(raw_df)
# Append start controls to physiology samples    
raw_df = data_helper.copy_start_control(raw_df, META_DATA)
# Sort data
raw_df = raw_df.sort_values(by = ["EXPERIMENT_NAME", "TIME", "SAMPLE_NAME", "BIO_REP", "TEC_REP"], ignore_index = True)
raw_df.head(3)

,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,ZETA_POTENTIAL_MV
0,CH230130,SC,A.obliquus,0,1,1,-19.20
1,CH230130,SC,A.obliquus,0,1,2,-18.40
2,CH230130,SC,A.obliquus,0,1,3,-17.10


# Custom Cleaning

In [5]:
clean_df = pd.DataFrame(raw_df)

# Drop all blank measurements
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"].str.contains("-Blank")].index, inplace = True)

# Lower species count from 7 to 5
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "C.vulgaris"].index, inplace = True)
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "P.duplex"].index, inplace = True)

print(f"{raw_df.shape[0] - clean_df.shape[0]} rows and {raw_df.shape[1] - clean_df.shape[1]} cols were cleaned.")

18 rows and 0 cols were cleaned.


# File Export

In [6]:
# Export to raw data to csv-file
export_df = pd.DataFrame(clean_df)
file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Zeta-Potential_Raw-Data.csv")
export_df.to_csv(file_name, encoding = "utf-8", index = False)

# Plot Visualization 

In [7]:
# Filter time before plotting
plot_df = pd.DataFrame(data_helper.filter(clean_df, time = [12]))

# Iterate through all experiment, visualize and save plots
for exp in data_helper.get_experiment_types(META_DATA, CONFIG):
    sub_df = plot_df[plot_df["EXPERIMENT_NAME"] == exp]
    sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
    
    plot_helper.visualize_boxplot(sub_df, sub_meta, # current data subset
                                  show_x_label = True,
                                  y_data = "ZETA_POTENTIAL_MV", 
                                  y_label = "Zeta Potential [mV]", # Without unit serves as file name
                                  y_ticks = [-45, 10, 5],
                                  y_labelpad = 12,)

Boxplot "Species Composition" created and saved.


# ARCHIVE